# Study B - Stimuli and Seed/Slot builder, Mini-Pilot Slot Builder

In [ ]:
# Study B stimuli builder (Google Colab)
# - Mount Drive
# - Load recipe_masterlist.csv + studyA_dialogs_from_goodbad.csv
# - Merge on recipe_title == title
# - Export studyB_stimuli.csv with requested columns

!pip -q install pandas

from google.colab import drive
drive.mount("/content/drive")

import ast
import re
import os
import pandas as pd

Mounted at /content/drive


## Consolidating relevant recipe information

In [ ]:
# config
ALL_RECIPE_PATH = "/content/drive/MyDrive/Masterarbeit/all_recipe.csv"
MASTERLIST_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/recipe_masterlist.csv"
OUT_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/all_recipe_filtered_by_masterlist.csv"

ALL_TITLE_COL = "recipe_name"   # in all_recipe.csv
MASTER_TITLE_COL = "title"      # in recipe_masterlist.csv
ING_COL = "ingredients_list"    # in both files

In [ ]:
def norm_title(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
         .str.lower()
    )

def norm_ing_list(x) -> str:
    """
    Normalisiert ingredients_list zu einem stabilen String-Key.
    Erwartet Formate wie "['salt', 'pepper']".
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""

    s = str(x).strip()

    # Attempts to parse Python list literals
    try:
        lst = ast.literal_eval(s)
        if isinstance(lst, list):
            lst = [str(i).strip().lower() for i in lst]
            # Normalise whitespace
            lst = [re.sub(r"\s+", " ", i) for i in lst]
            # as tuple-string key
            return "|".join(lst)
    except Exception:
        pass

    # Fallback
    s = s.strip("[]").replace('"', "'")
    s = re.sub(r"\s+", " ", s).lower()
    return s

In [ ]:
# load
df_all = pd.read_csv(ALL_RECIPE_PATH)
df_master = pd.read_csv(MASTERLIST_PATH)

# checks
for col in [ALL_TITLE_COL, ING_COL]:
    if col not in df_all.columns:
        raise ValueError(f"Missing column in all_recipe_filtered_by_masterlist.csv: {col}")

for col in [MASTER_TITLE_COL, ING_COL]:
    if col not in df_master.columns:
        raise ValueError(f"Missing column in recipe_masterlist.csv: {col}")

# normalize keys
df_all["_title_key"] = norm_title(df_all[ALL_TITLE_COL])
df_master["_title_key"] = norm_title(df_master[MASTER_TITLE_COL])

df_all["_ing_key"] = df_all[ING_COL].apply(norm_ing_list)
df_master["_ing_key"] = df_master[ING_COL].apply(norm_ing_list)

# strict match on (title, ingredients)
master_pairs = set(zip(df_master["_title_key"], df_master["_ing_key"]))
df_all["_keep"] = list(zip(df_all["_title_key"], df_all["_ing_key"]))
df_out = df_all[df_all["_keep"].isin(master_pairs)].copy()

# ensure 1 row per master entry (dedupe)
# If duplicates still exist (e.g., identical rows), keep the one with most non-null values
df_out["_non_nulls"] = df_out.notna().sum(axis=1)

df_out = (
    df_out.sort_values(["_title_key", "_ing_key", "_non_nulls"], ascending=[True, True, False])
          .drop_duplicates(subset=["_title_key", "_ing_key"], keep="first")
          .copy()
)

# cleanup
df_out = df_out.drop(columns=["_title_key", "_ing_key", "_keep", "_keep", "_non_nulls"], errors="ignore")

df_out.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(df_out)} rows to: {OUT_PATH}")

Saved 40 rows to: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/all_recipe_filtered_by_masterlist.csv


In [ ]:
# Paths
PATH_RECIPE_MASTERLIST = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/all_recipe_filtered_by_masterlist.csv"
PATH_STUDYA_DIALOGS    = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_dialogs_from_goodbad.csv"

OUT_DIR  = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB"
OUT_PATH = os.path.join(OUT_DIR, "studyB_stimuli_information.csv")
os.makedirs(OUT_DIR, exist_ok=True)

print("Masterlist filtered Recipes:", PATH_RECIPE_MASTERLIST)
print("StudyA dialogs:", PATH_STUDYA_DIALOGS)
print("Output:", OUT_PATH)

# Load CSVs
df_recipes = pd.read_csv(PATH_RECIPE_MASTERLIST)
df_dialogs = pd.read_csv(PATH_STUDYA_DIALOGS)

# Keep only relevant columns
DIALOG_COLS = ["dialog_id", "recipe_title", "recipe_type", "condition", "dialog_text"]

RECIPE_COLS = ["recipe_name", "carbohydrates", "fat", "kcal", "protein", "servings", "recipe_description", "recipe_directions", "ingredients_list"]

missing_d = [c for c in DIALOG_COLS if c not in df_dialogs.columns]
missing_r = [c for c in RECIPE_COLS if c not in df_recipes.columns]

if missing_d:
    raise ValueError(f"Missing columns in studyA_dialogs_from_goodbad.csv: {missing_d}\n"
                     f"Available: {list(df_dialogs.columns)}")
if missing_r:
    raise ValueError(f"Missing columns in recipe_masterlist.csv: {missing_r}\n"
                     f"Available: {list(df_recipes.columns)}")

df_dialogs = df_dialogs[DIALOG_COLS].copy()
df_recipes = df_recipes[RECIPE_COLS].copy()

# Normalize titles for matching (robust against whitespace/case)
def norm_title(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
         .str.lower()
    )

df_dialogs["_title_key"] = norm_title(df_dialogs["recipe_title"])
df_recipes["_title_key"] = norm_title(df_recipes["recipe_name"])

# sanity check: duplicates
dup_dialog_titles = df_dialogs["_title_key"].value_counts()
dup_recipe_titles = df_recipes["_title_key"].value_counts()

print("\n[Info] Dialog title keys:", len(dup_dialog_titles), "unique")
print("[Info] Recipe title keys:", len(dup_recipe_titles), "unique")

# Merge (left join: keep all dialogs)
df_merged = df_dialogs.merge(
    df_recipes.drop(columns=["recipe_name"]),  # keep recipe_title from dialogs as main title
    on="_title_key",
    how="left",
)

# Report match quality
n_total = len(df_merged)
n_matched = df_merged["fat"].notna().sum()
n_unmatched = n_total - n_matched

print(f"\n[Merge] Total dialog rows: {n_total}")
print(f"[Merge] Matched rows:      {n_matched}")
print(f"[Merge] Unmatched rows:    {n_unmatched}")

if n_unmatched > 0:
    print("\n[Warning] Unmatched recipe_title examples (first 10):")
    print(df_merged.loc[df_merged["fat"].isna(), "recipe_title"].dropna().unique()[:10])

# after df_merged = df_dialogs.merge(...)
df_merged = df_merged.rename(columns={
    "carbohydrates": "carbs",
    "kcal": "calories",
    "recipe_description": "description",
    "recipe_directions": "directions",
})

# Build final Study B stimuli
FINAL_COLS = [
    "dialog_id",
    "recipe_title",
    "recipe_type",
    "condition",
    "dialog_text",
    "fat",
    "carbs",
    "calories",
    "protein",
    "servings",
    "description",
    "directions",
    "ingredients_list",
]

df_studyB = df_merged[FINAL_COLS].copy()

# enforce numeric types for macros (keeps NaN if missing)
for c in ["fat", "carbs", "calories", "protein"]:
    df_studyB[c] = pd.to_numeric(df_studyB[c], errors="coerce")

# Save
df_studyB.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"\nSaved: {OUT_PATH}")

# Preview
display(df_studyB.head(10))

Masterlist filtered Recipes: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/all_recipe_filtered_by_masterlist.csv
StudyA dialogs: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_dialogs_from_goodbad.csv
Output: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli_information.csv

[Info] Dialog title keys: 40 unique
[Info] Recipe title keys: 40 unique

[Merge] Total dialog rows: 80
[Merge] Matched rows:      80
[Merge] Unmatched rows:    0

Saved: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli_information.csv


,dialog_id,recipe_title,recipe_type,condition,dialog_text,fat,carbs,calories,protein,servings,description,directions,ingredients_list
0,D001,Moong Dal,savory,good,Fat: Moong Dal has low fat content. Carb: Most...,1.16685,9.91822,62,3.54917,6,'I actually learned this in the kitchen of thi...,"Rinse split peas, add to saucepan with 2 1/2 c...","['salt', 'vegetable oil', 'water', 'fresh ging..."
1,D002,Bourbon Wieners,sweet,good,Fat: Bourbon Wieners have a notable fat conten...,16.71360,19.60390,290,6.66030,10,'Great appetizer. Family favorite for many yea...,"Place hot dogs, bourbon whiskey, ketchup, and ...","['brown sugar', 'ketchup', 'bourbon', 'hot dog']"
2,D003,Cobb Salad Ham Roll-ups,savory,good,Fat: The fat content seems relatively high due...,16.81090,4.61133,197,7.88013,4,'These fun rolled appetizers are a breeze to a...,Combine mayonnaise and mustard in a medium bow...,"['mayonnaise', 'hard cooked egg', 'ham', 'must..."
3,D004,Grain-Free Apple Cinnamon Dutch Babies,sweet,good,Fat: This recipe has a moderate fat content fr...,13.11520,13.35510,192,6.47763,8,"'A fast, easy and delicious gluten-free, grain...",Preheat oven to 425 degrees F (220 degrees C)....,"['vanilla extract', 'egg', 'unsalted butter', ..."
4,D005,Best Ever Muffins,sweet,good,Fat: The fat content seems relatively low in t...,8.77514,46.53960,284,5.17106,12,"'Start with this basic recipe, and add one of ...",Preheat oven to 400 degrees F (205 degrees C)....,"['flour', 'egg', 'white sugar', 'salt', 'veget..."
5,D006,Jeanine's Decadent Brownies,sweet,good,Fat: I think the fat content is relatively hig...,27.81200,49.60510,457,5.81147,24,'This brownie recipe is from my mom. I have ma...,Preheat oven to 350 degrees F (175 degrees C)....,"['flour', 'white sugar', 'salt', 'vegetable oi..."
6,D007,Red Bliss Potato Salad with Gorgonzola and Wal...,sweet,good,Fat: I like the use of olive oil in this recip...,7.24339,14.59600,132,3.09391,6,'Mayo-free potato salad combines new potatoes ...,Preheat an oven to 275 degrees F (135 degrees ...,"['salt', 'olive oil', 'gorgonzola cheese', 'fr..."
7,D008,Bread Pudding I,sweet,good,Fat: I see the fat content is relatively low. ...,2.36210,18.48950,118,6.02742,6,"'A delicious, easy to make Bread Pudding. Enjoy!'",Preheat oven to 325 degrees F (165 degrees C)....,"['vanilla extract', 'egg', 'egg white', 'honey..."
8,D009,Whole Grain Barley and Apple Porridge,sweet,good,"Fat: This recipe has a low fat content, mostly...",3.92033,10.45420,79,1.85700,4,'A tasty alternative from steel-cut oats. This...,"Place barley in a bowl and add 1 cup water, so...","['salt', 'ground cinnamon', 'water', 'butter',..."
9,D010,Coriander and Cumin Rubbed Pork Chops,savory,good,Fat: I think the fat content is relatively hea...,15.82730,3.45324,200,11.15110,2,'Chops rubbed with a simple but flavorful spic...,"Mix the salt, cumin, coriander, garlic, and 1 ...","['salt', 'garlic', 'ground pepper', 'ground cu..."


## Build Final Stimuli for Study B

In [ ]:
# Build final Study B stimuli file from TOP pool pairs

# paths
PATH_STIMULI_INFO = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli_information.csv"
PATH_TOP_POOL     = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_top_pool_pairs.csv"

OUT_STIMULI_FINAL = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli.csv"

# load
df_info = pd.read_csv(PATH_STIMULI_INFO)
df_top  = pd.read_csv(PATH_TOP_POOL)

# normalize dialog_id strings
for df in [df_info, df_top]:
    for c in ["dialog_id", "dialog_id_good", "dialog_id_bad"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

# basic column checks
required_info_cols = [
    "dialog_id","recipe_title","recipe_type","condition","dialog_text",
    "fat","carbs","calories","protein", "servings", "description","directions","ingredients_list"
]
missing_info = [c for c in required_info_cols if c not in df_info.columns]
if missing_info:
    raise ValueError(f"Missing columns in studyB_stimuli_information.csv: {missing_info}\n"
                     f"Available: {list(df_info.columns)}")

if "dialog_id_good" not in df_top.columns or "dialog_id_bad" not in df_top.columns:
    raise ValueError(f"TOP pool file must contain dialog_id_good and dialog_id_bad.\n"
                     f"Available: {list(df_top.columns)}")

# collect required dialog ids from top pool
ids_good = df_top["dialog_id_good"].dropna().astype(str).str.strip().tolist()
ids_bad  = df_top["dialog_id_bad"].dropna().astype(str).str.strip().tolist()
top_ids = sorted(set(ids_good + ids_bad))

print("Top pool pairs:", len(df_top))
print("Top pool unique dialog_ids needed:", len(top_ids))

# filter info table
df_final = df_info[df_info["dialog_id"].isin(top_ids)].copy()

# sanity: check missing IDs
found_ids = set(df_final["dialog_id"].astype(str))
missing_ids = [did for did in top_ids if did not in found_ids]

print("Rows matched:", len(df_final))
if missing_ids:
    print("\n Missing dialog_ids in stimuli_information (first 20):", missing_ids[:20])
    raise ValueError(f"{len(missing_ids)} dialog_ids from top_pool were not found in stimuli_information. "
                     f"Fix by ensuring both files use the same dialog_id scheme and no whitespace/format issues.")

# enforce condition normalization
df_final["condition"] = df_final["condition"].astype(str).str.lower().str.strip()

# sanity: per recipe should be exactly 2 dialogs (good+bad)
per_recipe = df_final.groupby("recipe_title")["condition"].apply(lambda s: set(s)).reset_index(name="cond_set")
bad_recipes = per_recipe[~per_recipe["cond_set"].apply(lambda s: ("good" in s) and ("bad" in s))]

if not bad_recipes.empty:
    print("\n Some recipes do NOT have both conditions in final stimuli:")
    display(bad_recipes)
    # not raising hard error, because might still want to inspect manually
else:
    print("All recipes have both good+bad in final stimuli.")

# order rows: by recipe_title, condition (good first), dialog_id
cond_order = {"good": 0, "bad": 1}
df_final["_cond_order"] = df_final["condition"].map(cond_order).fillna(99).astype(int)

df_final = df_final.sort_values(["recipe_title", "_cond_order", "dialog_id"]).drop(columns=["_cond_order"])

# keep required columns
df_final = df_final[required_info_cols].copy()

# save
df_final = df_final.reset_index(drop=True)
df_final.to_csv(OUT_STIMULI_FINAL, index=True, encoding="utf-8")
print("Final stimuli rows:", len(df_final))
display(df_final.head(10))

Top pool pairs: 17
Top pool unique dialog_ids needed: 34
Rows matched: 34
All recipes have both good+bad in final stimuli.
Final stimuli rows: 34


,dialog_id,recipe_title,recipe_type,condition,dialog_text,fat,carbs,calories,protein,servings,description,directions,ingredients_list
0,D040,Apple Pie Filling,sweet,good,Fat: This recipe has very little fat. Carb: Mo...,0.070120,23.42010,90,0.140240,40,"'Freezer apple pie filling. With this recipe, ...","In a large bowl, toss apples with lemon juice ...","['white sugar', 'salt', 'ground cinnamon', 'wa..."
1,D077,Apple Pie Filling,sweet,bad,"Fat: Ugh, too much sugar. Carb: Whatever, it's...",0.070120,23.42010,90,0.140240,40,"'Freezer apple pie filling. With this recipe, ...","In a large bowl, toss apples with lemon juice ...","['white sugar', 'salt', 'ground cinnamon', 'wa..."
2,D034,Au Gratin-Pumpkin Layered Casserole,savory,good,Fat: I think the fat content is relatively bal...,6.682380,4.81959,106,7.007630,6,"'In this Idahoan approach to lasagna, we've la...",Preheat oven to 350 degrees F.<br>Prepare Au G...,"['salt', 'ricotta cheese', 'egg', 'pepper', 'o..."
3,D064,Au Gratin-Pumpkin Layered Casserole,savory,bad,"Fat: Whatever, it's got some fat. Carb: I thin...",6.682380,4.81959,106,7.007630,6,"'In this Idahoan approach to lasagna, we've la...",Preheat oven to 350 degrees F.<br>Prepare Au G...,"['salt', 'ricotta cheese', 'egg', 'pepper', 'o..."
4,D002,Bourbon Wieners,sweet,good,Fat: Bourbon Wieners have a notable fat conten...,16.713600,19.60390,290,6.660300,10,'Great appetizer. Family favorite for many yea...,"Place hot dogs, bourbon whiskey, ketchup, and ...","['brown sugar', 'ketchup', 'bourbon', 'hot dog']"
5,D042,Bourbon Wieners,sweet,bad,"Fat: Ugh, hot dogs have too much fat. Carb: Wh...",16.713600,19.60390,290,6.660300,10,'Great appetizer. Family favorite for many yea...,"Place hot dogs, bourbon whiskey, ketchup, and ...","['brown sugar', 'ketchup', 'bourbon', 'hot dog']"
6,D019,Carrot Casserole,savory,good,Fat: The Carrot Casserole has a significant am...,11.252000,10.56590,151,2.744390,6,'Great side dish for big meals. I make it ever...,Preheat oven to 350 degrees F (175 degrees C)....,"['salt and pepper', 'butter', 'condensed cream..."
7,D060,Carrot Casserole,savory,bad,"Fat: Whatever, it's got some fat. Carb: I thin...",11.252000,10.56590,151,2.744390,6,'Great side dish for big meals. I make it ever...,Preheat oven to 350 degrees F (175 degrees C)....,"['salt and pepper', 'butter', 'condensed cream..."
8,D037,Carrot and Sweet Potato Tzimmes,savory,good,"Fat: This recipe has a very low fat content, m...",0.223501,20.69620,83,0.894005,12,'This is a favorite from my childhood - my mot...,Preheat oven to 350 degrees F (175 degrees C)....,"['salt', 'ground cinnamon', 'cornstarch', 'ora..."
9,D071,Carrot and Sweet Potato Tzimmes,savory,bad,"Fat: Whatever, it's got some fat. Carb: No, it...",0.223501,20.69620,83,0.894005,12,'This is a favorite from my childhood - my mot...,Preheat oven to 350 degrees F (175 degrees C)....,"['salt', 'ground cinnamon', 'cornstarch', 'ora..."


## Assignment Seed for Study B

In [ ]:
# assignment seed for Study B

!pip -q install pandas numpy

import numpy as np
import pandas as pd
from pathlib import Path

# Paths
BASE_DIR = Path("/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB")

INPUT_STIMULI = BASE_DIR / "studyB_stimuli.csv"
TOP_PAIRS_FILE = BASE_DIR / "studyB_top_pool_pairs.csv"

OUTPUT_SEED = BASE_DIR / "studyB_assignment_seed.csv"

print("Stimuli :", INPUT_STIMULI)
print("TopPairs:", TOP_PAIRS_FILE, "(optional)")
print("Output  :", OUTPUT_SEED)

# Build assignment seed (4-per-dialog + balanced extras)
REQUIRED_COLS = ["dialog_id", "condition", "recipe_title"]

def _read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

def build_studyB_assignment_seed(
    stimuli_path: Path,
    out_path: Path,
    base_slots: int = 4,
    extra_recipes: int = 3,
    top_pairs_path: Path | None = None,
    choose_extras_by: str = "smallest_human_delta_overall",
) -> pd.DataFrame:
    """
    Creates studyB_assignment_seed.csv with remaining_slots/initial_slots.

    Default quota logic:
    - Every dialog gets base_slots (default 4)
    - Select extra_recipes recipes and add +1 to BOTH dialogs (good+bad) of that recipe

    Extras selection:
    - If top_pairs_path exists, choose recipes by smallest delta (more fragile manipulation -> more precision)
    - Else: choose first N recipes alphabetically (fallback)
    """
    df = _read_csv(stimuli_path)

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"studyB_stimuli.csv is missing required columns: {missing}\n"
            f"Found columns: {list(df.columns)}"
        )

    seed = df[REQUIRED_COLS].copy()

    # Normalize text columns
    for col in ["dialog_id", "condition", "recipe_title"]:
        seed[col] = seed[col].astype(str).str.strip()

    # keep only valid dialog_ids
    seed = seed[seed["dialog_id"].notna() & (seed["dialog_id"] != "")].copy()

    # ensure uniqueness (each dialog once)
    if seed["dialog_id"].duplicated().any():
        dups = int(seed["dialog_id"].duplicated().sum())
        print(f"[WARN] {dups} duplicate dialog_id rows — keeping first occurrence.")
        seed = seed.drop_duplicates(subset=["dialog_id"], keep="first").copy()

    # sanity: each recipe should have exactly one good + one bad in the final stimuli set
    seed["condition"] = seed["condition"].str.lower()
    bad_cond = seed[~seed["condition"].isin(["good", "bad"])]
    if not bad_cond.empty:
        raise ValueError(f"Found non good/bad conditions in stimuli: {bad_cond['condition'].unique().tolist()}")

    recipe_counts = seed.groupby("recipe_title")["condition"].nunique()
    not_two = recipe_counts[recipe_counts != 2]
    if len(not_two) > 0:
        print("[WARN] Some recipes do not have exactly 2 conditions in studyB_stimuli.csv (expected good+bad).")
        print(not_two.head(20))

    # base slots
    seed["remaining_slots"] = int(base_slots)
    seed["initial_slots"] = int(base_slots)

    # choose extra recipes (balanced: +1 for both dialogs)
    extra_recipe_titles = []

    if top_pairs_path is not None and Path(top_pairs_path).exists():
        pairs = _read_csv(Path(top_pairs_path))

        # expected columns in pairs
        need = ["recipe_title"]
        if choose_extras_by == "smallest_human_delta_overall":
            need += ["human_delta_overall"]
        elif choose_extras_by == "smallest_llm_delta_overall":
            need += ["llm_delta_overall"]
        else:
            raise ValueError("choose_extras_by must be 'smallest_human_delta_overall' or 'smallest_llm_delta_overall'")

        miss = [c for c in need if c not in pairs.columns]
        if miss:
            raise ValueError(f"Top pairs file missing columns {miss}. Found: {list(pairs.columns)}")

        pairs["recipe_title"] = pairs["recipe_title"].astype(str).str.strip()

        # pick recipes that actually exist in stimuli
        pairs = pairs[pairs["recipe_title"].isin(set(seed["recipe_title"]))].copy()

        sort_col = "human_delta_overall" if choose_extras_by == "smallest_human_delta_overall" else "llm_delta_overall"
        pairs[sort_col] = pd.to_numeric(pairs[sort_col], errors="coerce")

        pairs = pairs.dropna(subset=[sort_col]).sort_values(sort_col, ascending=True)
        extra_recipe_titles = pairs["recipe_title"].drop_duplicates().head(int(extra_recipes)).tolist()

        if len(extra_recipe_titles) < int(extra_recipes):
            print(f"[WARN] Could only pick {len(extra_recipe_titles)} extra recipes from top_pairs.")
    else:
        # fallback: alphabetical (still balanced, just not "principled")
        extra_recipe_titles = sorted(seed["recipe_title"].unique().tolist())[:int(extra_recipes)]
        print("[WARN] top_pairs file not found -> choosing extra recipes alphabetically:", extra_recipe_titles)

    # apply extras: +1 for both dialogs of each selected recipe
    if extra_recipe_titles:
        mask = seed["recipe_title"].isin(extra_recipe_titles)
        seed.loc[mask, "remaining_slots"] = seed.loc[mask, "remaining_slots"] + 1
        seed.loc[mask, "initial_slots"] = seed.loc[mask, "initial_slots"] + 1

    # reorder
    seed = seed[["dialog_id", "recipe_title", "condition", "remaining_slots", "initial_slots"]].copy()

    # save
    out_path.parent.mkdir(parents=True, exist_ok=True)
    seed.to_csv(out_path, index=False, encoding="utf-8")

    print("\n[OK] Wrote:", out_path)
    print("[INFO] dialogs:", len(seed))
    print("[INFO] total slots:", int(seed["initial_slots"].sum()))
    print("[INFO] good slots:", int(seed.loc[seed["condition"] == "good", "initial_slots"].sum()))
    print("[INFO] bad slots :", int(seed.loc[seed["condition"] == "bad", "initial_slots"].sum()))
    if extra_recipe_titles:
        print("[INFO] extra recipes (+1/+1):", extra_recipe_titles)

    return seed

seed = build_studyB_assignment_seed(
    stimuli_path=INPUT_STIMULI,
    out_path=OUTPUT_SEED,
    base_slots=4,
    extra_recipes=7,
    top_pairs_path=TOP_PAIRS_FILE,
    choose_extras_by="smallest_human_delta_overall",
)

display(seed.head(34))
display(seed.groupby(["condition"])["initial_slots"].sum().reset_index())
display(seed.groupby(["recipe_title"])["initial_slots"].sum().describe())


def _stable_int_hash(s: str) -> int:
    """Stable hash for deterministic seeding across sessions."""
    h = 2166136261
    for ch in str(s):
        h ^= ord(ch)
        h = (h * 16777619) & 0xFFFFFFFF
    return int(h)

def assign_next_dialog_blocked_by_recipe(
    seed_df: pd.DataFrame,
    participant_id: str,
    recipe_col: str = "recipe_title",
    condition_col: str = "condition",
    remaining_col: str = "remaining_slots",
) -> tuple[str, pd.DataFrame]:
    """
    Study Plan assignment:
    - Blocking by recipe (stratification): pick a recipe first
    - Randomization within recipe: pick good/bad within that recipe
    - Quotas enforced via remaining_slots (decrement after assignment)

    Returns: (dialog_id, updated_seed_df)
    """
    d = seed_df.copy()

    # normalize + numeric
    d[recipe_col] = d[recipe_col].astype(str).str.strip()
    d[condition_col] = d[condition_col].astype(str).str.lower().str.strip()
    d["dialog_id"] = d["dialog_id"].astype(str).str.strip()
    d[remaining_col] = pd.to_numeric(d[remaining_col], errors="coerce").fillna(0).astype(int)

    # only available rows
    avail = d[d[remaining_col] > 0].copy()
    if avail.empty:
        raise RuntimeError("No remaining slots left in assignment seed.")

    rng = np.random.default_rng(_stable_int_hash(participant_id))

    # pick recipe block, weighted by remaining slots in that recipe
    per_recipe = avail.groupby(recipe_col)[remaining_col].sum()
    recipes = per_recipe.index.to_numpy()
    weights = per_recipe.to_numpy(dtype=float)
    weights = weights / weights.sum()
    chosen_recipe = rng.choice(recipes, p=weights)

    # within recipe: pick condition/dialog weighted by remaining slots
    sub = avail[avail[recipe_col] == chosen_recipe].copy()
    # sanity: expect exactly 2 rows (good+bad) but allow robust behavior
    w2 = sub[remaining_col].to_numpy(dtype=float)
    w2 = w2 / w2.sum()
    chosen_idx = rng.choice(sub.index.to_numpy(), p=w2)

    chosen_dialog = str(d.loc[chosen_idx, "dialog_id"])

    # decrement
    d.loc[chosen_idx, remaining_col] = int(d.loc[chosen_idx, remaining_col]) - 1

    return chosen_dialog, d

Stimuli : /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli.csv
TopPairs: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_top_pool_pairs.csv (optional)
Output  : /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_assignment_seed.csv

[OK] Wrote: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_assignment_seed.csv
[INFO] dialogs: 34
[INFO] total slots: 150
[INFO] good slots: 75
[INFO] bad slots : 75
[INFO] extra recipes (+1/+1): ['White Chocolate and Passion Fruit Cheesecake', 'Minted Marinated Zucchini', 'Au Gratin-Pumpkin Layered Casserole', 'Sweet and Nutty Pork Chops', 'Whole Grain Barley and Apple Porridge', 'Cuban-Style Pork and Sweet Potatoes', 'Red Bliss Potato Salad with Gorgonzola and Walnuts']


,dialog_id,recipe_title,condition,remaining_slots,initial_slots
0,D040,Apple Pie Filling,good,4,4
1,D077,Apple Pie Filling,bad,4,4
2,D034,Au Gratin-Pumpkin Layered Casserole,good,5,5
3,D064,Au Gratin-Pumpkin Layered Casserole,bad,5,5
4,D002,Bourbon Wieners,good,4,4
5,D042,Bourbon Wieners,bad,4,4
6,D019,Carrot Casserole,good,4,4
7,D060,Carrot Casserole,bad,4,4
8,D037,Carrot and Sweet Potato Tzimmes,good,4,4
9,D071,Carrot and Sweet Potato Tzimmes,bad,4,4


,condition,initial_slots
0,bad,75
1,good,75


,initial_slots
count,17.000000
mean,8.823529
std,1.014599
min,8.000000
25%,8.000000
50%,8.000000
75%,10.000000
max,10.000000


## Assignment Seed for Mini-Pilot Study B

In [ ]:
# Assignment Seed for Mini-Pilot

!pip -q install pandas numpy

import numpy as np
import pandas as pd
from pathlib import Path

# Paths
BASE_DIR = Path("/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB")

INPUT_STIMULI = BASE_DIR / "studyB_stimuli.csv"
TOP_PAIRS_FILE = BASE_DIR / "studyB_top_pool_pairs.csv"

OUTPUT_SEED = BASE_DIR / "studyB_pilot_assignment_seed.csv"

print("Stimuli :", INPUT_STIMULI)
print("TopPairs:", TOP_PAIRS_FILE)
print("Output  :", OUTPUT_SEED)

# Mini-Pilot Seed (4-per-dialog, 17 pairs, N=36)
REQUIRED_COLS = ["dialog_id", "condition", "recipe_title"]

def _read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

def build_pilot_assignment_seed(
    stimuli_path: Path,
    top_pairs_path: Path,
    out_path: Path,
    top_x: int = 17,
    slots_per_dialog: int = 4,  # 2 Pairs/Person → 36 Personen → ~4 Ratings pro Pair → 4 Ratings pro Dialog
) -> pd.DataFrame:
    df = _read_csv(stimuli_path)
    pairs = _read_csv(top_pairs_path)

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"studyB_stimuli.csv missing columns: {missing}")

    required_pairs_cols = ["recipe_title", "dialog_id_good", "dialog_id_bad", "pass_precheck"]
    missing_pairs = [c for c in required_pairs_cols if c not in pairs.columns]
    if missing_pairs:
        raise ValueError(f"Top pairs file missing columns: {missing_pairs}")

    # robust precheck filter
    pairs["pass_precheck"] = pairs["pass_precheck"].astype(str).str.lower().isin(["true", "1", "yes"])
    pairs = pairs[pairs["pass_precheck"]].copy()

    # rank-based top_x
    if "rank_score" in pairs.columns:
        pairs["rank_score"] = pd.to_numeric(pairs["rank_score"], errors="coerce")
        pairs = pairs.sort_values("rank_score", ascending=False).head(top_x).copy()
    else:
        pairs = pairs.head(top_x).copy()

    pairs = pairs.reset_index(drop=True)
    print(f"[INFO] Using {len(pairs)} recipe pairs for pilot (top_x={top_x})")

    needed_dialog_ids = set(
        pairs["dialog_id_good"].astype(str).str.strip().tolist()
        + pairs["dialog_id_bad"].astype(str).str.strip().tolist()
    )

    seed = df[df["dialog_id"].astype(str).str.strip().isin(needed_dialog_ids)].copy()
    seed = seed[REQUIRED_COLS].copy()

    for col in ["dialog_id", "condition", "recipe_title"]:
        seed[col] = seed[col].astype(str).str.strip()

    seed["condition"] = seed["condition"].str.lower()
    seed = seed.drop_duplicates(subset=["dialog_id"], keep="first")

    found_ids = set(seed["dialog_id"])
    missing_ids = needed_dialog_ids - found_ids
    if missing_ids:
        raise ValueError(f"Missing dialog_ids in stimuli: {sorted(missing_ids)}")

    seed["remaining_slots"] = int(slots_per_dialog)
    seed["initial_slots"] = int(slots_per_dialog)

    seed = seed[["dialog_id", "recipe_title", "condition", "remaining_slots", "initial_slots"]].copy()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    seed.to_csv(out_path, index=False, encoding="utf-8")

    print("\n[OK] Wrote:", out_path)
    print(f"[INFO] Total dialogs: {len(seed)} (expected {len(pairs) * 2})")
    print(f"[INFO] Total slots: {int(seed['initial_slots'].sum())} (expected {len(seed) * slots_per_dialog})")
    return seed

# Mini-Pilot Seed
pilot_seed = build_pilot_assignment_seed(
    stimuli_path=INPUT_STIMULI,
    top_pairs_path=TOP_PAIRS_FILE,
    out_path=OUTPUT_SEED,
    top_x=18,
    slots_per_dialog=4,
)

display(pilot_seed.head(10))
display(pilot_seed.groupby(["condition"])["initial_slots"].sum().reset_index())
display(pilot_seed.groupby(["recipe_title"])["initial_slots"].sum().describe())


# Deterministic allocation for Mini-Pilot (no randomisation required)

def get_pilot_assignment(
    pilot_seed_df: pd.DataFrame,
    rater_id: str,
) -> tuple[str, str, str]:
    """
    Deterministic mapping for Mini-Pilot.

    Returns: (dialog_id, recipe_title, condition)

    Logic:
    - pair_index = stable_hash(rater_id) % 17
    - Within the pair: rater_id determines good-first or bad-first
    - However: Mini-Pilot displays BOTH dialogs, so only the pair is needed
    """
    d = pilot_seed_df.copy()

    # Stable hash for deterministic mapping
    def stable_hash(s: str) -> int:
        h = 2166136261
        for ch in str(s):
            h ^= ord(ch)
            h = (h * 16777619) & 0xFFFFFFFF
        return int(h)

    # All available recipes
    recipes = sorted(d["recipe_title"].unique())
    n_recipes = len(recipes)

    # Deterministic index
    rater_hash = stable_hash(rater_id)
    pair_index = rater_hash % n_recipes

    chosen_recipe = recipes[pair_index]

    # Get both dialogs for this recipe
    recipe_dialogs = d[d["recipe_title"] == chosen_recipe]

    if len(recipe_dialogs) != 2:
        raise ValueError(f"Expected 2 dialogs for recipe {chosen_recipe}, found {len(recipe_dialogs)}")

    # For Mini-Pilot: you need the whole pair, not just a single dialogue
    # App.py then decides between "good-first" and "bad-first"
    return chosen_recipe, pair_index, recipe_dialogs

Stimuli : /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_stimuli.csv
TopPairs: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_top_pool_pairs.csv
Output  : /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_pilot_assignment_seed.csv
[INFO] Using 17 recipe pairs for pilot (top_x=18)

[OK] Wrote: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_pilot_assignment_seed.csv
[INFO] Total dialogs: 34 (expected 34)
[INFO] Total slots: 136 (expected 136)


,dialog_id,recipe_title,condition,remaining_slots,initial_slots
0,D040,Apple Pie Filling,good,4,4
1,D077,Apple Pie Filling,bad,4,4
2,D034,Au Gratin-Pumpkin Layered Casserole,good,4,4
3,D064,Au Gratin-Pumpkin Layered Casserole,bad,4,4
4,D002,Bourbon Wieners,good,4,4
5,D042,Bourbon Wieners,bad,4,4
6,D019,Carrot Casserole,good,4,4
7,D060,Carrot Casserole,bad,4,4
8,D037,Carrot and Sweet Potato Tzimmes,good,4,4
9,D071,Carrot and Sweet Potato Tzimmes,bad,4,4


,condition,initial_slots
0,bad,68
1,good,68


,initial_slots
count,17.0
mean,8.0
std,0.0
min,8.0
25%,8.0
50%,8.0
75%,8.0
max,8.0
